In [ ]:
# =========================
# Imports
# =========================

# --- Standard library ---
import base64
import json
import os
import re
from datetime import datetime
from io import BytesIO

# --- Third-party ---
import requests
from PIL import Image
from dotenv import load_dotenv
from IPython.display import Markdown, display
from rich import print as rp

# --- Local / project ---
import tools
import utils
import llm


# =========================
# Environment & Client
# =========================
load_dotenv()


marketing research agent

In [ ]:
def market_research_agent(return_messages: bool = False):

    utils.log_agent_title_html("Market Research Agent", "🕵️‍♂️")

    prompt_ = f"""
You are a fashion market research agent tasked with preparing a trend analysis for a summer sunglasses campaign.

Your goal:
1. Explore current fashion trends related to sunglasses using web search.
2. Review the internal product catalog to identify items that align with those trends.
3. Recommend one or more products from the catalog that best match emerging trends.
4. If needed, today date is {datetime.now().strftime("%Y-%m-%d")}.

You can call the following tools:
- tavily_search_tool: to discover external web trends.
- product_catalog_tool: to inspect the internal sunglasses catalog.

Once your analysis is complete, summarize:
- The top 2–3 trends you found.
- The product(s) from the catalog that fit these trends.
- A justification of why they are a good fit for the summer campaign.
"""
    messages = [{"role": "user", "content": prompt_}]
    tools_ = tools.get_available_tools()

    while True:
        print("running the while loop")
        msg = llm.agent( model="nvidia/nemotron-3-super-120b-a12b:free",messages=messages,)

        if msg["content"]:
            rp(msg["content"])
            return (msg["content"], messages) if return_messages else msg["content"]

        if msg["tool_calls"]:
            for tool_call in msg["tool_calls"]:
                result = tools.get_tool_response(tool_call)

                messages.append(msg)
                messages.append(result)
        else:
            utils.log_unexpected_html()
            return ("[⚠️ Unexpected: No tool_calls or content returned]", messages) if return_messages else "[⚠️ Unexpected: No tool_calls or content returned]"

In [ ]:
market_research_result = market_research_agent()

graphic design agent

In [ ]:
def graphic_designer_agent(trend_insights: str, caption_style: str = "short punchy", size: str = "1024x1024") -> dict:

    """
    Args:
        trend_insights (str): Trend summary from the researcher agent.
        caption_style (str): Optional style hint for caption.
        size (str): Image resolution (e.g., '1024x1024').

    Returns:
        dict: A dictionary with image_url, prompt, and caption.
    """

    utils.log_agent_title_html("Graphic Designer Agent", "🎨")

    # Step 1: Generate prompt 
    system_message = (
        "You are a visual marketing assistant. Based on the input trend insights, "
        "write a creative and visual prompt for an AI image generation model, and also a short caption."
    )

    user_prompt = f"""
Trend insights:
{trend_insights}

Please output:
1. A vivid, descriptive prompt to guide image generation.
2. A marketing caption in style: {caption_style}.

Respond in this format:
{{"prompt": "...", "caption": "..."}}
"""
    messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt}
        ]
    
    msg = llm.agent( model="openai/gpt-oss-120b:free",messages=messages,)

    content = msg["content"]
    match = re.search(r'\{.*\}', content, re.DOTALL)
    parsed = json.loads(match.group(0)) if match else {"error": "No JSON returned", "raw": content}

    prompt = parsed["prompt"]
    caption = parsed["caption"]

    return {
        "prompt": prompt,
        "caption": caption
    }

    # Step 2: Generate image directly using openai-python
    # openai_client = openai.OpenAI()

    # image_response = openai_client.images.generate(
    #     model="dall-e-3",
    #     prompt=prompt,
    #     size=size,
    #     quality="standard",
    #     n=1,
    #     response_format="url"
    # )

    # image_url = image_response.data[0].url

    # # Save image locally
    # img_bytes = requests.get(image_url).content
    # img = Image.open(BytesIO(img_bytes))

    # filename = os.path.basename(image_url.split("?")[0])
    # image_path = filename
    # img.save(image_path)


    # # Log summary with local image
    # utils.log_final_summary_html(f"""
    #     <h3>Generated Image and Caption</h3>

    #     <p><strong>Image Path:</strong> <code>{image_path}</code></p>

    #     <p><strong>Generated Image:</strong></p>
    #     <img src="{image_path}" alt="Generated Image" style="max-width: 100%; height: auto; border: 1px solid #ccc; border-radius: 8px; margin-top: 10px; margin-bottom: 10px;">

    #     <p><strong>Prompt:</strong> {prompt}</p>
    # """)


    # return {
    #     "image_url": image_url,
    #     "prompt": prompt,
    #     "caption": caption,
    #     "image_path": image_path  
    # }



In [ ]:
graphic_designer_agent_result = graphic_designer_agent(
    trend_insights=market_research_result,
)


In [ ]:
rp(graphic_designer_agent_result)

copywrite agent

In [ ]:
def copywriter_agent(image_prompt: str, trend_summary: str) -> dict:

    """
    send an image prompt and a trend summary and return a campaign quote.

    Args:
        image_prompt (str): prompt of the image to be analyzed.
        trend_summary (str): Text from the researcher agent.
        model (str): OpenAI model (e.g., openai:o4-mini, openai:gpt-4o)

    Returns:
        dict: {
            "quote": "...",
            "justification": "...",
            "image_prompt": "..."
        }
    """

    utils.log_agent_title_html("Copywriter Agent", "✍️")

    # Step 2: Build message
    messages = [
        {
            "role": "system",
            "content": "You are a copywriter that creates elegant campaign quotes based on an image prompt and a marketing trend summary."
        },
        {
            "role": "user",
            "content": 
                f"""
Here is a visual marketing image prompt and a trend analysis:

Image prompt: 
\"\"\"{image_prompt}\"\"\"

Trend summary:
\"\"\"{trend_summary}\"\"\"

Please return a JSON object like:
{{
  "quote": "A short, elegant campaign phrase (max 12 words)",
  "justification": "Why this quote matches the image and trend"
}}

DO NOT MAKE TOOL CALLS ALL THE INFORMATION ARE PROVIDED"""
        }
    ]

    msg = llm.agent(model="nvidia/nemotron-3-super-120b-a12b:free" , messages=messages)

    # Step 4: Parse JSON response
    content = msg["content"]

    rp(content)

    try:
        match = re.search(r'\{.*\}', content, re.DOTALL)
        parsed = json.loads(match.group(0)) if match else {"error": "No valid JSON returned"}
    except Exception as e:
        parsed = {"error": f"Failed to parse: {e}", "raw": content}


    parsed["image_prompt"] = image_prompt
    return parsed


In [ ]:
copywriter_agent_result = copywriter_agent(
    image_prompt=graphic_designer_agent_result["prompt"],
    trend_summary=market_research_result,
)

packaging agent

In [ ]:
def packaging_agent(trend_summary: str, image_prompt: str, quote: str, justification: str, output_path: str = "campaign_summary.md") -> str:

    """
    Packages the campaign assets into a beautifully formatted markdown report for executive review.

    Args:
        trend_summary (str): Summary of the market trends.
        image_prompt (str): prompt of the campaign image.
        quote (str): Marketing quote to overlay.
        justification (str): Explanation for the quote.
        output_path (str): Path to save the markdown report.

    Returns:
        str: Path to the saved markdown file.
    """

    utils.log_agent_title_html("Packaging Agent", "📦")

    messages=[
            {"role": "system", "content": "You are a marketing communication expert writing elegant campaign summaries for executives."},
            {"role": "user", "content": f"""
        Please rewrite the following trend summary to be clear, professional, and engaging for a CEO audience:

        \"\"\"{trend_summary.strip()}\"\"\"

        DO NOT MAKE TOOL CALLS ALL THE INFORMATION ARE PROVIDED
        """}
                ]
    
    msg = llm.agent(model="nvidia/nemotron-3-super-120b-a12b:free" , messages=messages )

    rp(msg)
    
    beautified_summary = msg["content"]

    # Combine all parts into markdown
    markdown_content = f"""# 🕶️ Summer Sunglasses Campaign – Executive Summary

## 📊 Refined Trend Insights
{beautified_summary}

## 🎯 Prompt for Campaign Visual
{image_prompt}

## ✍️ Campaign Quote
{quote.strip()}

## ✅ Why This Works
{justification.strip()}

---

*Report generated on {datetime.now().strftime('%Y-%m-%d')}*
"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)

    return output_path



In [ ]:
packaging_agent_result = packaging_agent(
    trend_summary=market_research_result,
    image_prompt=graphic_designer_agent_result["prompt"],
    quote=copywriter_agent_result["quote"],
    justification=copywriter_agent_result["justification"],
    output_path=f"campaign_summary_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.md"
)

full campaign pipeline

In [ ]:
def run_sunglasses_campaign_pipeline(output_path: str = "campaign_summary.md") -> dict:
    """
    Runs the full summer sunglasses campaign pipeline:
    1. Market research (search trends + match products)
    2. Generate visual + caption
    3. Generate quote based on image + trend
    4. Create executive markdown report

    Returns:
        dict: Dictionary containing all intermediate results + path to final report
    """
    # 1. Run market research agent
    trend_summary = market_research_agent()
    print("✅ Market research completed")

    # 2. Generate image prompt + caption
    visual_result = graphic_designer_agent(trend_insights=trend_summary)
    image_prompt = visual_result["prompt"]
    print("🖼️ Image generated")

    # 3. Generate quote based on image + trends
    quote_result = copywriter_agent(image_prompt=image_prompt, trend_summary=trend_summary)
    quote = quote_result.get("quote", "")
    justification = quote_result.get("justification", "")
    print("💬 Quote created")

    # 4. Generate markdown report
    md_path = packaging_agent(
        trend_summary=trend_summary,
        image_prompt=image_prompt,  
        quote=quote,
        justification=justification,
        output_path=f"campaign_summary_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.md"
    )

    print(f"📦 Report generated: {md_path}")

    return {
        "trend_summary": trend_summary,
        "visual": visual_result,
        "quote": quote_result,
        "markdown_path": md_path
    }


In [ ]:
results = run_sunglasses_campaign_pipeline()